# Simulation Robustness Testing

This notebook evaluates how robust the DoH detection classifier is under controlled flow-feature perturbations. The goal is to test whether small changes in timing, packet size, and packet rate features reduce the model’s ability to detect malicious DoH traffic.

In [ ]:
import pandas as pd
import numpy as np
import joblib

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

np.random.seed(42)

DATA_PATH = "labeled_doh_dataset.csv"
MODEL_PATH = "detector_model.joblib"  # update once model is available
LABEL_COL = "Label"

## Load and Prepare Data

We load the labeled flow-feature dataset, remove invalid values, drop identifier columns, and separate features from labels.

In [ ]:
df = pd.read_csv(DATA_PATH)

drop_cols = [
    "src_ip", "dst_ip", "src_mac", "dst_mac",
    "src_port", "dst_port", "timestamp", "flow_id"
]

df = df.replace([np.inf, -np.inf, "Infinity", "-Infinity"], np.nan)
df = df.dropna()
df = df.drop(columns=[col for col in drop_cols if col in df.columns], errors="ignore")

X = df.drop(columns=[LABEL_COL])
y = df[LABEL_COL]

print("Dataset shape:", df.shape)
print("Feature shape:", X.shape)
print(y.value_counts())
X.head()

## Load Model and Evaluation Helper

The trained classifier is loaded from a serialized `.joblib` file. The evaluation helper reports core performance metrics for each simulation scenario, allowing us to compare how performance changes under different traffic perturbations.

NOTE: Replace MODEL_PATH with the actual file name once received

In [ ]:
try:
    model = joblib.load(MODEL_PATH)
    print(f"Loaded model from {MODEL_PATH}")
except FileNotFoundError:
    model = None
    print(f"Model not found at {MODEL_PATH}. Add the trained .joblib file before running evaluation.")


def evaluate(model, X_eval, y_eval, scenario):
    if model is None:
        return {
            "scenario": scenario,
            "accuracy": np.nan,
            "precision": np.nan,
            "recall": np.nan,
            "f1": np.nan,
            "note": "model not loaded"
        }

    preds = model.predict(X_eval)

    return {
        "scenario": scenario,
        "accuracy": accuracy_score(y_eval, preds),
        "precision": precision_score(y_eval, preds, pos_label="Malicious", zero_division=0),
        "recall": recall_score(y_eval, preds, pos_label="Malicious", zero_division=0),
        "f1": f1_score(y_eval, preds, pos_label="Malicious", zero_division=0),
        "false_negatives": int(((y_eval == "Malicious") & (preds == "Benign")).sum())
    }

Columns in dataset:


NameError: name 'df' is not defined

## Perturbation Experiments

We define controlled feature-level simulations that modify timing, packet size, and packet rate features. These represent realistic changes an attacker could make to reduce the distinctiveness of malicious DoH traffic.

In [ ]:
def perturb_columns(X_input, keywords, mode="noise", noise_level=0.2, scale_factor=1.25):
    X_mod = X_input.copy()

    selected_cols = [
        col for col in X_mod.columns
        if any(keyword in col.lower() for keyword in keywords)
    ]

    for col in selected_cols:
        if mode == "noise":
            noise = np.random.normal(loc=1.0, scale=noise_level, size=len(X_mod))
            noise = np.clip(noise, 0.1, None)
            X_mod[col] = X_mod[col] * noise
        elif mode == "scale":
            X_mod[col] = X_mod[col] * scale_factor

    return X_mod, selected_cols


experiments = {}

experiments["Original Traffic"] = X

experiments["Timing Perturbation"], timing_cols = perturb_columns(
    X,
    keywords=["iat", "duration", "time"],
    mode="noise",
    noise_level=0.20
)

experiments["Packet Size Perturbation"], size_cols = perturb_columns(
    X,
    keywords=["len", "byte", "byts", "totlen", "pkt_size"],
    mode="scale",
    scale_factor=1.25
)

experiments["Packet Rate Perturbation"], rate_cols = perturb_columns(
    X,
    keywords=["pkts_s", "pkt_s", "rate"],
    mode="noise",
    noise_level=0.30
)

X_combined = X.copy()
X_combined, _ = perturb_columns(X_combined, ["iat", "duration", "time"], mode="noise", noise_level=0.20)
X_combined, _ = perturb_columns(X_combined, ["len", "byte", "byts", "totlen", "pkt_size"], mode="scale", scale_factor=1.25)
X_combined, _ = perturb_columns(X_combined, ["pkts_s", "pkt_s", "rate"], mode="noise", noise_level=0.30)

experiments["Combined Perturbation"] = X_combined

print("Timing columns:", timing_cols)
print("Packet size columns:", size_cols)
print("Packet rate columns:", rate_cols)

## Robustness Evaluation

Each perturbation is evaluated against the same labels to measure how much classifier performance changes under simulated traffic variation.

In [ ]:
results = []

for scenario, X_eval in experiments.items():
    results.append(evaluate(model, X_eval, y, scenario))

results_df = pd.DataFrame(results)
results_df

## Save Results

In [ ]:
results_df.to_csv("robustness_results.csv", index=False)
print("Saved robustness_results.csv")